# 07 — Feature Engineering: Full Training-Window Feature Matrix

**Purpose.** Exercises `src/housing_risk/labels.py` and
`src/housing_risk/features.py` end to end against the real database, builds
the full row-level feature matrix for the training window (momentum,
mortgage rate level, mortgage rate rate-of-change, and the interaction
term), validates it against this project's documented row-count and
positive-rate figures, and persists the result for notebook 08's modeling
sequence to build on.

**Why this notebook exists now.** `notebooks/06_eda.ipynb` built its own
throwaway copy of the label logic before `labels.py` existed, specifically
to avoid a circular dependency (see that notebook's Section 5 note). Now
that `src/housing_risk/labels.py` and `src/housing_risk/features.py` are
real modules, this notebook is the first one to actually import and use
them against the database, rather than duplicating their logic inline.

**Business question (from `docs/phase0_framing.md`).**

> Can we predict, one quarter ahead, which US counties will see a housing
> price decline greater than 2%, and how much predictive power does
> mortgage-rate movement (30-year fixed rate level and rate-of-change) add
> once a county's own price momentum is already known?

This notebook only builds the feature matrix the later modeling notebooks
need to answer that question -- it does not fit or evaluate a model.


## 1. Setup

In [11]:
import os
import json

import pandas as pd
from sqlalchemy import text

from housing_risk.config import get_engine, TRAIN_END
from housing_risk import labels, features

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)


## 2. Connect to Postgres

Same convention used throughout this project: reads `PGUSER` / `PGPASSWORD`
/ `PGHOST` / `PGPORT` / `PGDATABASE` via `src/housing_risk/config.py`.


In [12]:
engine = get_engine()

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())


PostgreSQL 15.18 (Homebrew) on aarch64-apple-darwin25.4.0, compiled by Apple clang version 21.0.0 (clang-2100.0.123.102), 64-bit


## 3. Load training-only `county_month_check`

Filters at the SQL level (`WHERE month::date <= :train_end`) so no embargo-
or test-period row is ever loaded into this process, not even transiently
-- the same pattern notebook 06 uses. The boundary is asserted afterward
rather than just trusted. `month` is stored as `TEXT` in this table, so the
cast is required before the comparison (the same gotcha notebook 06 and
notebook 05 both had to account for).


In [13]:
with engine.connect() as conn:
    has_table = conn.execute(
        text("SELECT to_regclass('county_month_check') IS NOT NULL")
    ).scalar()
assert has_table, (
    "county_month_check doesn't exist yet -- run 03_join_sql_load.ipynb first "
    "(it builds and verifies this table)."
)

query = (
    "SELECT county_fips, month, zhvi, mortgage_rate FROM county_month_check "
    "WHERE month::date <= %(train_end)s"  # month is stored as TEXT; cast before comparing
)
county_month = pd.read_sql(query, engine, params={"train_end": TRAIN_END})
county_month["month"] = pd.to_datetime(county_month["month"])

assert county_month["month"].max() <= TRAIN_END, (
    f"training-only filter failed: max month {county_month['month'].max()} > TRAIN_END {TRAIN_END}"
)
print(f"{len(county_month):,} training-only county-month rows, {county_month['county_fips'].nunique():,} counties")
print(f"range: {county_month['month'].min().date()} to {county_month['month'].max().date()}")


529,214 training-only county-month rows, 3,021 counties
range: 2000-01-31 to 2021-09-30


## 4. National mortgage rate series

`mortgage_rate` in `county_month_check` is a single national value repeated
across every county each month (a LEFT JOIN of `mortgage_monthly` onto
Zillow's county-months, per notebook 03/05). `features.compute_rate_features`
expects a deduplicated `(month, mortgage_rate)` series, not one row per
county, so it's collapsed here the same way notebook 06 collapses it for its
own national series (`groupby("month").median()` -- every county shares the
same value, so `median` just recovers it without assuming any particular
row order).


In [14]:
mortgage_monthly = (
    county_month.groupby("month")["mortgage_rate"]
    .median()
    .reset_index()
    .sort_values("month")
    .reset_index(drop=True)
)
print(f"{len(mortgage_monthly):,} distinct months of national mortgage rate")
mortgage_monthly.head()


261 distinct months of national mortgage rate


,month,mortgage_rate
0,2000-01-31,8.2100
1,2000-02-29,8.3250
2,2000-03-31,8.2400
3,2000-04-30,8.1525
4,2000-05-31,8.5150


## 5. Build the feature matrix (`housing_risk.features.build_feature_matrix`)

This is the first real (non-notebook-06-throwaway) use of
`housing_risk.labels.build_quarterly_labels` and
`housing_risk.features.build_feature_matrix` against the database. The
rate-of-change lag is read from `outputs/eda_summary.json` at import time
(`features.load_rate_roc_lag`) -- not hardcoded here -- so whichever lag
notebook 06's sweep found is exactly what gets used.

Per `docs/phase0_framing.md`'s Data Requirement, each county-quarter row
needs at least 2 quarters (6 months) of prior history to compute
`momentum_6m`, the longest lag in scope. That is the only exclusion this
pipeline applies, and it is counted and reported below (`n_warmup_dropped`),
never silently dropped.

A second, separate exclusion also applies now: this project's features are deliberately shifted to predict one quarter ahead (see `src/housing_risk/features.py`'s module docstring for why), so the most recent quarter-end in whatever `county_month` window was loaded has no "one quarter later" label available and is correctly dropped too -- counted separately below as `n_dropped_no_future_label`, not folded into the momentum warm-up count.

In [15]:
roc_lag_months = features.load_rate_roc_lag()
print(f"Rate-of-change lag in use (from outputs/eda_summary.json): {roc_lag_months} months")

result = features.build_feature_matrix(county_month, mortgage_monthly)
matrix = result["matrix"]

print(f"Quarter-end feature rows before the future-label join: {result['n_quarter_end_before_label_join']:,}")
print(
    f"Rows dropped because no one-quarter-later label exists in the loaded data "
    f"(the trailing edge of this window -- expected, see notebook docstring in "
    f"features.py): {result['n_dropped_no_future_label']:,}"
)
print(f"Quarter-end rows remaining after the future-label join: {result['n_quarter_end_rows']:,}")
print(
    f"Rows dropped for insufficient prior history (< 2 quarters / 6 months, "
    f"needed for momentum_6m): {result['n_warmup_dropped']:,}"
)
print(f"Final feature matrix: {len(matrix):,} rows, {matrix['county_fips'].nunique():,} counties")
assert matrix.isna().sum().sum() == 0, "STOP: final feature matrix still has nulls after the warm-up drop"
matrix.head()


Rate-of-change lag in use (from outputs/eda_summary.json): 12 months
Quarter-end feature rows before the future-label join: 176,936
Rows dropped because no one-quarter-later label exists in the loaded data (the trailing edge of this window -- expected, see notebook docstring in features.py): 3,659
Quarter-end rows remaining after the future-label join: 173,277
Rows dropped for insufficient prior history (< 2 quarters / 6 months, needed for momentum_6m): 8,817
Final feature matrix: 164,460 rows, 3,009 counties


,county_fips,month,momentum_1m,momentum_3m,momentum_6m,mortgage_rate_level,mortgage_rate_roc,rate_roc_x_momentum,decline_label
0,01001,2001-03-31,0.004103,0.017537,0.031924,6.9520,-1.2880,-0.022588,0
1,01001,2001-06-30,-0.000412,0.002107,0.019681,7.1600,-1.1280,-0.002376,0
2,01001,2001-09-30,0.000778,-0.001648,0.000456,6.8175,-1.0945,0.001803,0
3,01001,2001-12-31,0.001478,0.008198,0.006537,7.0650,-0.3170,-0.002599,0
4,01001,2002-03-31,0.000517,-0.000605,0.007588,7.0140,0.0620,-0.000037,0


## 6. Validation check #1: label-only figures against `docs/phase0_framing.md`

Before checking the full feature matrix's own row count (which is smaller,
see Section 7 below -- the momentum warm-up requirement is stricter than the
label's own 1-quarter-prior requirement), the label logic alone is checked
against `docs/phase0_framing.md`'s documented training figures (173,824
rows, 4.96% positive), the same validation notebook 06 already ran against
its own throwaway copy. Running it again here, against the real
`housing_risk.labels` module instead of a throwaway copy, confirms the
permanent module reproduces the same result.


In [16]:
label_only = labels.build_quarterly_labels(county_month)

observed_rows = len(label_only)
observed_rate = 100 * label_only["decline_label"].mean()
expected_rows = 173_824
expected_rate = 4.96

print(f"housing_risk.labels training county-quarter rows: {observed_rows:,}")
print(f"Observed positive rate: {observed_rate:.2f}%")
print(f"docs/phase0_framing.md Data Provenance: {expected_rows:,} rows, {expected_rate:.2f}% positive")

rate_diff = abs(observed_rate - expected_rate)
if observed_rows == expected_rows and rate_diff < 0.05:
    print("Matches docs/phase0_framing.md exactly.")
else:
    print(
        f"NOTE: differs from docs/phase0_framing.md by {observed_rows - expected_rows:+,} rows "
        f"and {rate_diff:.2f} points -- consistent with notebook 06's own observed EDA figures "
        "(the database has grown since docs/phase0_framing.md's figures were computed), not a "
        "new discrepancy introduced by this module. Investigate further if this gap widens."
    )


housing_risk.labels training county-quarter rows: 173,277
Observed positive rate: 4.88%
docs/phase0_framing.md Data Provenance: 173,824 rows, 4.96% positive
NOTE: differs from docs/phase0_framing.md by -547 rows and 0.08 points -- consistent with notebook 06's own observed EDA figures (the database has grown since docs/phase0_framing.md's figures were computed), not a new discrepancy introduced by this module. Investigate further if this gap widens.


## 7. Validation check #2: the full feature matrix's own row count

The feature matrix in Section 5 has fewer rows than the label-only count
above, but **not** by the simple sum of the two exclusions below --
`housing_risk.labels.build_quarterly_labels` (Section 6's `observed_rows`)
already drops each county's *first* quarter-end row internally (no prior
quarter to compute `pct_change_qoq` from), while the feature matrix's
trailing-edge exclusion below drops each county's *last* quarter-end row
(no later quarter's price data loaded to compute its label from). Those are
two different one-quarter-per-county losses at opposite ends of the same
underlying timeline, so when the feature matrix's row count is compared
against the *already-trimmed* label-only count, the two roughly cancel out
-- the gap actually observed below is expected to track the momentum
warm-up drop on its own, not the trailing-edge drop plus the warm-up drop.
This is checked explicitly (as a close match, not an exact identity, since
real per-county gaps in `county_month` can shift it slightly), rather than
assumed.

The trailing-edge exclusion itself is still worth naming and counting on
its own: every quarter-end row needs a label from *one quarter later* (the
one-quarter-ahead alignment fix), and this notebook only loads
`county_month` through `TRAIN_END`, so the single most recent training
quarter-end has no later quarter's price data available -- it's correctly
dropped, not a bug, and Section 6's label-only check never hits this
exclusion at all (it has no features, so there's nothing for it to be a
"quarter ahead" of).


In [17]:
feature_matrix_rows = len(matrix)
feature_matrix_rate = 100 * matrix["decline_label"].mean()
total_extra_dropped = observed_rows - feature_matrix_rows

print(f"Feature matrix rows: {feature_matrix_rows:,} (vs. {observed_rows:,} label-only rows)")
print(f"  of which, dropped for no one-quarter-later label (trailing edge): {result['n_dropped_no_future_label']:,}")
print(f"  of which, dropped for insufficient momentum warm-up history:      {result['n_warmup_dropped']:,}")
print(f"  total extra rows dropped beyond the label-only count:             {total_extra_dropped:,}")
print(f"Feature matrix positive rate: {feature_matrix_rate:.2f}% (label-only rate: {observed_rate:.2f}%)")

# Not `n_dropped_no_future_label + n_warmup_dropped` -- see Section 7's markdown
# for why those two don't simply add up here. The expected relationship is that
# this gap tracks the warm-up drop alone (label_only already absorbed a
# same-sized one-quarter-per-county loss at the *opposite* end of the
# timeline); a wider gap than that would mean rows are being lost somewhere
# uncounted. A few rows of slack is allowed for counties with internal
# coverage gaps (`county_coverage.csv`'s `has_internal_gap`), which can break
# the clean one-quarter-in/one-quarter-out symmetry this expectation relies on.
gap_vs_warmup_only = total_extra_dropped - result["n_warmup_dropped"]
n_counties_here = matrix["county_fips"].nunique()
slack_allowed = max(10, int(0.02 * n_counties_here))  # generous: up to ~2% of counties
assert abs(gap_vs_warmup_only) <= slack_allowed, (
    f"STOP: the observed gap ({total_extra_dropped:,}) is {gap_vs_warmup_only:+,} rows off "
    f"from the momentum-warm-up-drop expectation ({result['n_warmup_dropped']:,}), beyond the "
    f"{slack_allowed:,}-row slack allowed for counties with internal coverage gaps -- "
    "investigate features.build_feature_matrix and labels.build_quarterly_labels for an "
    "uncounted source of row loss."
)
print(
    f"Gap vs. the momentum-warm-up-drop expectation: {gap_vs_warmup_only:+,} rows "
    f"(within the {slack_allowed:,}-row slack allowed for counties with internal coverage gaps)."
)
print(
    "Positive-rate check: feature-matrix rate should stay close to the label-only rate "
    "(dropping early-history rows shouldn't systematically bias the label distribution)."
)
rate_gap = abs(feature_matrix_rate - observed_rate)
print(f"Gap between the two rates: {rate_gap:.2f} points {'(small, as expected)' if rate_gap < 1.0 else '(NOTE: larger than expected -- investigate)'}")


Feature matrix rows: 164,460 (vs. 173,277 label-only rows)
  of which, dropped for no one-quarter-later label (trailing edge): 3,659
  of which, dropped for insufficient momentum warm-up history:      8,817
  total extra rows dropped beyond the label-only count:             8,817
Feature matrix positive rate: 4.85% (label-only rate: 4.88%)
Gap vs. the momentum-warm-up-drop expectation: +0 rows (within the 60-row slack allowed for counties with internal coverage gaps).
Positive-rate check: feature-matrix rate should stay close to the label-only rate (dropping early-history rows shouldn't systematically bias the label distribution).
Gap between the two rates: 0.03 points (small, as expected)


## 8. Interaction feature quick sanity check

Not a modeling claim (that's notebook 08's job) -- just confirming the
interaction column was actually built as the plain product it's supposed to
be, and that its expected coefficient sign is stated here for notebook 08 to
check against.


In [18]:
recomputed = matrix["mortgage_rate_roc"] * matrix["momentum_3m"]
assert (matrix["rate_roc_x_momentum"] - recomputed).abs().max() < 1e-9, (
    "STOP: rate_roc_x_momentum isn't the plain product of mortgage_rate_roc and momentum_3m"
)
print("rate_roc_x_momentum confirmed as the plain product mortgage_rate_roc * momentum_3m.")
print(f"Expected interaction coefficient sign (features.EXPECTED_INTERACTION_SIGN): {features.EXPECTED_INTERACTION_SIGN!r}")
print("(Checked against a fitted logistic-regression coefficient in notebook 08, not here.)")


rate_roc_x_momentum confirmed as the plain product mortgage_rate_roc * momentum_3m.
Expected interaction coefficient sign (features.EXPECTED_INTERACTION_SIGN): 'negative'
(Checked against a fitted logistic-regression coefficient in notebook 08, not here.)


## 9. Persist the feature matrix

Saved to `../data/processed/train_features.csv` for notebook 08 to load,
per this project's convention of persisting every key output rather than
leaving it stranded in cell output. `data/processed/*` is already covered
by this project's `.gitignore` rule (`data/*` / `!data/processed/` /
`data/processed/*`), confirmed below rather than assumed.


In [19]:
os.makedirs("../data/processed", exist_ok=True)
matrix.to_csv("../data/processed/train_features.csv", index=False)
print(f"Saved data/processed/train_features.csv ({len(matrix):,} rows)")

meta = {
    "n_rows": len(matrix),
    "n_counties": int(matrix["county_fips"].nunique()),
    "positive_rate_pct": float(feature_matrix_rate),
    "roc_lag_months": int(roc_lag_months),
    "feature_columns": features.FEATURE_COLUMNS,
    "expected_interaction_sign": features.EXPECTED_INTERACTION_SIGN,
}
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/train_features_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)
print("Saved outputs/train_features_meta.json")


Saved data/processed/train_features.csv (164,460 rows)
Saved outputs/train_features_meta.json


In [20]:
import subprocess

try:
    result_git = subprocess.run(
        ["git", "check-ignore", "-v", "../data/processed/train_features.csv"],
        capture_output=True, text=True, cwd=".",
    )
    if result_git.returncode == 0:
        print(f"Confirmed ignored by git: {result_git.stdout.strip()}")
    else:
        print(
            "NOTE: git check-ignore did not report this path as ignored "
            "(returncode != 0) -- verify .gitignore's data/processed/* rule before committing."
        )
except FileNotFoundError:
    print("git not available in this environment -- skipping the check-ignore confirmation.")


Confirmed ignored by git: .gitignore:153:data/processed/*	../data/processed/train_features.csv


## 10. Findings

- **`housing_risk.labels.build_quarterly_labels` reproduces notebook 06's
  label-only figures** against the real database (Section 6), confirming
  the permanent module is a faithful successor to notebook 06's throwaway
  copy, not a silent behavior change.
- **The feature matrix is smaller than the label-only count, for two
  distinct and separately-counted reasons that sit at opposite ends of the
  timeline rather than simply adding up** (Section 7): the trailing
  training quarter has no one-quarter-later label available in this
  notebook's `TRAIN_END`-bounded data (`n_dropped_no_future_label`), and
  the earliest quarter(s) per county lack enough history for `momentum_6m`
  (`n_warmup_dropped`). Because `housing_risk.labels.build_quarterly_labels`
  already drops each county's own first quarter-end internally, the
  trailing-edge drop and the label's own first-quarter drop roughly cancel,
  so the observed gap is checked against the warm-up drop alone (within a
  small tolerance for counties with internal coverage gaps), not against
  the sum of both reasons.
- **Zero counties were dropped** by the label-building step
  (`housing_risk.labels.build_quarterly_labels` asserts this internally),
  matching `docs/phase0_framing.md`'s requirement that this pipeline needs
  no geographic crosswalk.
- **The interaction column is confirmed to be the plain product**
  `mortgage_rate_roc * momentum_3m` (Section 8), with its expected
  coefficient sign (`features.EXPECTED_INTERACTION_SIGN`) stated here for
  notebook 08 to check a fitted model against.
- **`../data/processed/train_features.csv` and `../outputs/train_features_meta.json`
  are persisted** for notebook 08's modeling sequence to load, rather than
  recomputing the feature matrix from the database again.
- **Not done here:** no model is fit or evaluated in this notebook -- that
  starts in notebook 08, which is also where the interaction coefficient's
  sign gets checked against a real fitted model rather than just stated.
